# NeqSim CO2 Impurity Kinetics & Thermodynamics Interactive Guide

Welcome to the comprehensive guide for simulating **chemical reactions and phase behavior of trace impurities in dense-phase and supercritical CO2 transport streams** (ship and pipeline transport).

### Key Topics Covered in this Notebook:
1. **Model Initialization & Importing Framework**
2. **Selecting Equation of State (EOS) & Fluid Thermodynamics**
3. **Setting Impurity Levels, Water Content & Wall Materials**
4. **Running Dynamic ODE Simulations & Building Summary Tables**
   - **Table 1**: 10-Hour Species Concentration Time-Series Table
   - **Table 2**: Reaction Kinetics & Thermodynamic Equilibrium Summary Table ($E_a, K_{\text{eq}}, k_f, r_0$)
5. **Interactive Benchmark Test Cases Suite** (Cases 1–4, Oxidant-Free Streams, Pipeline Conditions & High H2S Streams)



## 1. Importing NeqSim Impurity Kinetics Framework


In [ ]:
import numpy as np
import pandas as pd
from neqsim_co2_kinetics import CO2ImpurityKineticsModel

print("CO2 Impurity Kinetics Engine successfully imported!")



## 2. Choosing Equation of State (EOS) & Calculating Fluid Density

The framework integrates with **NeqSim SRK EOS** or EOS dense fluid density correlations.
Fluid molar density $\rho_m$ (kmol/m³) is automatically calculated based on system pressure $P$ (bar) and temperature $T$ (K).

Let's test fluid properties at **Pipeline Transport Conditions** ($25^\circ\text{C}, 100\text{ bar}$) vs **Ship Transport Conditions** ($-25^\circ\text{C}, 25\text{ bar}$):



In [ ]:
# Pipeline Transport (25 °C, 100 bar)
model_pipe = CO2ImpurityKineticsModel(T_kelvin=298.15, P_bar=100.0, water_ppm=50.0)
print(f"Pipeline Molar Density (25 °C, 100 bar):   {model_pipe.molar_density:.2f} kmol/m³")

# Ship Transport (-25 °C, 25 bar)
model_ship = CO2ImpurityKineticsModel(T_kelvin=248.15, P_bar=25.0, water_ppm=50.0)
print(f"Ship Transport Molar Density (-25 °C, 25 bar): {model_ship.molar_density:.2f} kmol/m³")



## 3. Setting Impurity Levels & Choosing Wall Material

You can specify arbitrary trace impurity concentrations in **parts per million (ppm)**:
- `H2S`: Hydrogen Sulfide (ppm)
- `SO2`: Sulfur Dioxide (ppm)
- `NO2`: Nitrogen Dioxide (ppm)
- `O2`: Molecular Oxygen (ppm)
- `H2O`: Water Content (ppm)

### Wall Material Options:
- `'carbon_steel'` or `'magnetite'`: Catalyzes heterogeneous elemental sulfur formation ($R_8$: $E_{a, \text{S8}} = 42.0\text{ kJ/mol}$).
- `'stainless_steel'` or `'inert'`: Uncatalyzed surface ($E_{a, \text{S8}} = 65.0\text{ kJ/mol}$).



In [ ]:
# Define custom feed stream in ppm
custom_feed = {
    'H2S': 10.0,
    'SO2': 10.0,
    'NO2': 10.0,
    'O2': 10.0,
    'H2O': 10.0
}

# Select Carbon Steel / Magnetite Surface
model = CO2ImpurityKineticsModel(
    T_kelvin=248.15,   # -25 °C
    P_bar=25.0,        # 25 bar
    water_ppm=10.0,    # 10 ppm H2O
    material='carbon_steel'
)

print(f"Model initialized for material: '{model.material}' at T={model.T} K, P={model.P} bar")



## 4. Helper Functions for Generating Table 1 & Table 2

Below are standard helper functions to simulate any case and format both **Table 1 (Time-Series)** and **Table 2 (Reaction Kinetics & $K_{\text{eq}}$)**.



In [ ]:
def generate_tables_for_case(case_name, feed, T_K=248.15, P_bar=25.0, material='carbon_steel'):
    model = CO2ImpurityKineticsModel(T_kelvin=T_K, P_bar=P_bar, water_ppm=feed.get('H2O', 10.0), material=material)
    rho_m = model.molar_density
    
    # 1. Dynamic ODE Integration over 10 hours
    res = model.simulate(feed, duration_sec=10.0*3600.0, num_points=1001)
    t_h = res['time_hours']
    
    # Build Table 1 (Concentration Time-Series)
    target_hours = [0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0]
    table1_rows = []
    
    for target in target_hours:
        idx = np.argmin(np.abs(t_h - target))
        row = {
            'Time (h)': target,
            'H2S (ppm)': res['ppm']['H2S'][idx],
            'SO2 (ppm)': res['ppm']['SO2'][idx],
            'NO2 (ppm)': res['ppm']['NO2'][idx],
            'NO (ppm)': res['ppm']['NO'][idx],
            'O2 (ppm)': res['ppm']['O2'][idx],
            'H2O (ppm)': res['ppm']['H2O'][idx],
            'H2SO4 (ppm)': res['ppm']['H2SO4'][idx],
            'HNO3 (ppm)': res['ppm']['HNO3'][idx],
            'NH3 (ppm)': res['ppm']['NH3'][idx],
            'S8 (ppm)': res['ppm']['S8'][idx]
        }
        table1_rows.append(row)
        
    df_table1 = pd.DataFrame(table1_rows)
    
    # Build Table 2 (Reaction Kinetics & Keq Summary)
    rates_dict = model._calculate_pure_physical_rate_constants(feed.get('H2O', 10.0))
    reactions_info = [
        ("R1", "SO2 + 0.5 O2 + H2O <-> H2SO4",           "Direct Thermal SO2 Oxidation",  45.0, rates_dict['k1_f'],  rates_dict['Keq1']),
        ("R2", "H2S + 3 NO2 <-> SO2 + H2O + 3 NO",       "H2S Oxidation by NO2",          28.0, rates_dict['k2_f'],  rates_dict['Keq2']),
        ("R3a","SO2 + NO2 + H2O <-> NO + H2SO4",         "Base NO2 Oxidation (No H2S)",  26.0, rates_dict['k3a_f'], rates_dict['Keq3']),
        ("R3b","SO2 + H2S + NO2 + O2 -> H2SO4",          "Radical Chain Co-Catalysis",    15.0, rates_dict['k3b_f'], rates_dict['Keq3']),
        ("R4", "2 NO + O2 <-> 2 NO2",                    "NO Termolecular Re-Oxidation", -4.4, rates_dict['k4_f'],  rates_dict['Keq4']),
        ("R5", "3 NO2 + H2O <-> 2 HNO3 + NO",            "Reversible NO2 Hydrolysis",     28.0, rates_dict['k5_f'],  rates_dict['Keq5']),
        ("R6", "H2S + 1.5 O2 <-> SO2 + H2O",             "Uncatalyzed Direct H2S Oxidation", 65.0, rates_dict['k6_f'],rates_dict['Keq6']),
        ("R7", "5 H2S + 6 NO + 4 H2O -> 6 NH3 + 5 SO2",  "Trace Ammonia Reactions",       15.0, rates_dict['k7_f'],  1e10),
        ("R8", "H2S + 0.5 O2 -> 1/8 S8 + H2O",           "Catalytic S8 Formation (CS/Magnetite)", rates_dict['Ea8'], rates_dict['k8_f'], 1e10)
    ]
    
    C_H2S   = (feed.get('H2S', 1e-6) * 1e-6) * rho_m
    C_SO2   = (feed.get('SO2', 1e-6) * 1e-6) * rho_m
    C_NO2   = (feed.get('NO2', 1e-6) * 1e-6) * rho_m
    C_NO    = (feed.get('NO',  1e-6) * 1e-6) * rho_m
    C_O2    = (feed.get('O2',  1e-6) * 1e-6) * rho_m
    C_H2O   = (feed.get('H2O', 1e-6) * 1e-6) * rho_m
    
    table2_rows = []
    for rxn_id, eq, name, ea, k_f, keq in reactions_info:
        if rxn_id == "R1":
            r0 = k_f * C_SO2 * (C_O2**0.5) * C_H2O
        elif rxn_id == "R2":
            r0 = k_f * C_H2S * C_NO2
        elif rxn_id == "R3a":
            r0 = k_f * C_SO2 * C_NO2 * C_H2O
        elif rxn_id == "R3b":
            r0 = k_f * C_SO2 * (C_H2S**0.5) * C_NO2 * (C_O2**0.5)
        elif rxn_id == "R4":
            r0 = k_f * (C_NO**2) * C_O2
        elif rxn_id == "R5":
            r0 = k_f * (C_NO2**3) * C_H2O
        elif rxn_id == "R6":
            r0 = k_f * C_H2S * (C_O2**0.5)
        elif rxn_id == "R7":
            r0 = k_f * C_H2S * C_NO * C_H2O
        elif rxn_id == "R8":
            r0 = k_f * C_H2S * (C_O2**0.5)

        r0_ppm_hr = (r0 / rho_m) * 1e6 * 3600.0
        table2_rows.append({
            'ID': rxn_id,
            'Reaction Equation': eq,
            'Ea (kJ/mol)': ea,
            'Keq': f"{keq:.4e}",
            'k_f': f"{k_f:.4e}",
            'r0 (kmol/m3.s)': f"{r0:.4e}",
            'r0 (ppm/hr)': f"{r0_ppm_hr:.4e}"
        })
        
    df_table2 = pd.DataFrame(table2_rows)
    
    print("=" * 100)
    print(f"{case_name.upper()} (T={T_K - 273.15:.1f} °C, P={P_bar} bar, Material={material})")
    print("=" * 100)
    print("
TABLE 1: 10-HOUR SPECIES CONCENTRATION TIME-SERIES TABLE")
    display(df_table1)
    
    print("
TABLE 2: REACTION KINETICS & THERMODYNAMIC EQUILIBRIUM SUMMARY TABLE")
    display(df_table2)
    
    return df_table1, df_table2



## 5. Interactive Benchmark Test Cases Suite

Below you can run any of the test cases examined together by executing the corresponding cell:



### Example 1: Case 1 Baseline (With BOTH H2S & NO2 at -25 °C, 25 bar)


In [ ]:
feed_case1 = {'H2S': 10.0, 'SO2': 10.0, 'NO2': 10.0, 'O2': 10.0, 'H2O': 10.0}
df1_t1, df1_t2 = generate_tables_for_case("Case 1: Baseline with Both H2S & NO2", feed_case1, T_K=248.15, P_bar=25.0)



### Example 2: Case 2 (Without H2S at -25 °C, 25 bar)


In [ ]:
feed_case2 = {'H2S': 1e-6, 'SO2': 10.0, 'NO2': 10.0, 'O2': 10.0, 'H2O': 10.0}
df2_t1, df2_t2 = generate_tables_for_case("Case 2: Without H2S", feed_case2, T_K=248.15, P_bar=25.0)



### Example 3: Case 3 (Without NO2 at -25 °C, 25 bar)


In [ ]:
feed_case3 = {'H2S': 10.0, 'SO2': 10.0, 'NO2': 1e-6, 'O2': 10.0, 'H2O': 10.0}
df3_t1, df3_t2 = generate_tables_for_case("Case 3: Without NO2", feed_case3, T_K=248.15, P_bar=25.0)



### Example 4: Case 4 (High Water 675 ppm & High NO2 72 ppm at 25 °C, 100 bar)


In [ ]:
feed_case4 = {'H2O': 675.0, 'NO2': 72.0, 'SO2': 10.0, 'O2': 10.0, 'H2S': 1e-6}
df4_t1, df4_t2 = generate_tables_for_case("Case 4: High Water & NO2 Equilibrium Bound", feed_case4, T_K=298.15, P_bar=100.0)



### Example 5: Oxidant-Free Stream (100 ppm H2O & 100 ppm H2S at 25 °C, 100 bar)


In [ ]:
feed_oxidant_free = {'H2O': 100.0, 'H2S': 100.0, 'SO2': 1e-6, 'NO2': 1e-6, 'O2': 1e-6}
df_off_t1, df_off_t2 = generate_tables_for_case("Oxidant-Free Stream", feed_oxidant_free, T_K=298.15, P_bar=100.0)



### Example 6: Pipeline Transport Stream (11 ppm H2O, 69 ppm SO2, 33 ppm NO2, 180 ppm O2 at 25 °C, 70 bar)


In [ ]:
feed_pipe = {'H2O': 11.0, 'SO2': 69.0, 'NO2': 33.0, 'O2': 180.0, 'H2S': 1e-6}
df_pipe_t1, df_pipe_t2 = generate_tables_for_case("High-O2 Pipeline Transport Stream", feed_pipe, T_K=298.15, P_bar=70.0)



### Example 7: High H2S Pipeline Stream (60 ppm H2S, 10 ppm others at 25 °C, 100 bar)


In [ ]:
feed_high_h2s = {'H2S': 60.0, 'SO2': 10.0, 'NO2': 10.0, 'O2': 10.0, 'H2O': 10.0}
df_h2s_t1, df_h2s_t2 = generate_tables_for_case("High-H2S Pipeline Transport Stream", feed_high_h2s, T_K=298.15, P_bar=100.0)

